<a href="https://colab.research.google.com/github/CPTR295/Sample-LLMs/blob/main/Improving_Reasoning_with_Inference_Time_Scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install reasoning_from_scratch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 76.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nccl-cu12
    Found e

In [1]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()

# Use CPU for the first run of this chapter
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [4]:
from reasoning_from_scratch.ch03 import render_prompt

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [2]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache

def generate_text_stream_concat_flex(model,tokenizer,prompt,device,max_new_tokens,verbose=False,generate_func=None,**generate_kwargs):
  if generate_func is None:
    generate_func = generate_text_basic_stream_cache

  input_ids = torch.tensor(tokenizer.encode(prompt),device=device).unsqueeze(0)
  generated_ids = []
  for token in generate_func(model=model,token_ids=input_ids,max_new_tokens=max_new_tokens,eos_token_id=tokenizer.eos_token_id,**generate_kwargs):
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id.item())
    if verbose:
      print(tokenizer.decode(next_token_id.tolist()),end="",flush=True)
  return tokenizer.decode(generated_ids)

In [6]:
response = generate_text_stream_concat_flex(model,tokenizer,prompt,device,max_new_tokens=100,verbose=True,generate_func = generate_text_basic_stream_cache)

 \boxed{10}

In [9]:
#Chain of though prompting just add explain step by step at the end

prompt_cot = prompt + "\n\nExplain step by step"

response_cot = generate_text_stream_concat_flex(model,tokenizer,prompt_cot,device,max_new_tokens=2048,verbose=True,generate_func = generate_text_basic_stream_cache)

:
To solve the problem, we need to find the value of \( x \) such that half the value of \( 3x - 9 \) is equal to \( x + 37 \).

### Step 1: Set up the equation
According to the problem, half the value of \( 3x - 9 \) is equal to \( x + 37 \). This can be written as:
\[
\frac{1}{2}(3x - 9) = x + 37
\]

### Step 2: Eliminate the fraction
To eliminate the fraction, multiply both sides of the equation by 2:
\[
2 \cdot \frac{1}{2}(3x - 9) = 2(x + 37)
\]
Simplifying both sides:
\[
3x - 9 = 2x + 74
\]

### Step 3: Solve for \( x \)
Subtract \( 2x \) from both sides to isolate \( x \):
\[
3x - 2x - 9 = 74
\]
Simplify:
\[
x - 9 = 74
\]
Add 9 to both sides to solve for \( x \):
\[
x = 74 + 9
\]
\[
x = 83
\]

### Final Answer:
\[
\boxed{83}
\]

In [10]:
def scale_logits_by_temperature(logits,temperature):
  if temperature <=0:
    raise ValueError("Temperature must be positive")
  return logits / temperature

In [11]:
#Adding temperature to the text generation

from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode
def generate_text_temp_stream_cache(model,token_ids,max_new_tokens,eos_token_id=None,temperature=0.):
  model.eval()
  cache = KVCache(n_layers=model.cfg['n_layers'])
  model.reset_kv_cache()

  out = model(token_ids,cache=cache)[:,-1]
  for _ in range(max_new_tokens):
    orig_device = token_ids.device

    if temperature is None or temperature ==0.0:
      next_token = torch.argmax(out,dim=-1,keepdim=True)
    else:
      logits = scale_logits_by_temperature(out,temperature)
      probas = torch.softmax(logits,dim=-1)
      next_token = torch.multinomial(probas.cpu(),num_samples=1)
      next_token = next_token.to(orig_device)

    if eos_token_id is not None and torch.all(next_token == eos_token_id):
      break

    yield next_token
    out = model(next_token,cache=cache)[:,-1]

In [12]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_temp_stream_cache,
    temperature=1.1
)

 We begin with the equation \(\frac{1}{2}(3x - 9) = x + 37\).
First, distribute the \(\frac{1}{2}\) on the left-side of the equation to get \(3x - 9 = 2(x + 37)\).
Then simplify the right side of the equation by using the distributive property and combining like terms: \(3x - 9 = 2x + 74\).
Next, subtract \(2x\) from both sides of the equation to get \(x - 9 = 2x + 74\).
Next, subtract \(x\) from both sides of the equation to reduce the number of \(x\) terms on the right side to exactly 1: \(-9 = x + 74\).
Finally, subtract 74 from both sides of the equation to get \(x = -83\).
The value of \(x\) is \(\boxed{-83}\).

In [13]:
prompt

'You are a helpful math assistant.\nAnswer the question and write the final result on a new line as:\n\\boxed{ANSWER}\n\nQuestion:\nHalf the value of $3x-9$ is $x+37$. What is the value of $x$?\n\nAnswer:'

In [17]:
#Top P filtering function

def top_p_filter(probas,top_p):
  if top_p is None or top_p >=1.0:
    return probas
  sorted_probas,sorted_idx = torch.sort(probas,dim=1,descending=True)
  cumprobas = torch.cumsum(sorted_probas,dim=1)
  prefix = cumprobas - sorted_probas
  keep = prefix < top_p
  keep[:,0]=True

  keep_sorted = torch.where(keep,sorted_probas,torch.zeros_like(sorted_probas))
  filtered = torch.zeros_like(probas).scatter(1,sorted_idx,keep_sorted)
  denom = torch.sum(filtered,dim=1,keepdim=True).clamp_min(1e-12)
  return filtered / denom




In [21]:
@torch.inference_mode
def generate_text_top_p_stream_cache(model,token_ids,max_new_tokens,eos_token_id=None,temperature=0.,top_p=None):
  model.eval()
  cache = KVCache(n_layers=model.cfg['n_layers'])
  model.reset_kv_cache()

  out = model(token_ids,cache=cache)[:,-1]
  for _ in range(max_new_tokens):
    orig_device = token_ids.device

    if temperature is None or temperature ==0.0:
      next_token = torch.argmax(out,dim=-1,keepdim=True)
    else:
      logits = scale_logits_by_temperature(out,temperature)
      probas = torch.softmax(logits,dim=-1)
      probas = top_p_filter(probas,top_p)
      next_token = torch.multinomial(probas.cpu(), num_samples=1)
      next_token = next_token.to(orig_device)
    if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
      break

    yield next_token
    out = model(next_token,cache=cache)[:,-1]

In [22]:
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.5,
    top_p=0.8,
)

 \boxed{18}

In [23]:
#improve response accuraacy with self consistency

from reasoning_from_scratch.ch03 import extract_final_candidate
from collections import Counter

def self_consistency_vote(
    model, tokenizer, prompt, device,
    num_samples=10, temperature=0.8, top_p=0.9, max_new_tokens=2048,
    show_progress=True, show_long_answer=False, seed=None,
):
    full_answers, short_answers = [], []

    # 1) Sample multiple answers
    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model, tokenizer=tokenizer, prompt=prompt, device=device,
            max_new_tokens=max_new_tokens, verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature, top_p=top_p,
        )

        # 2) Extract the final (short) answer from each answer
        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    # 3) Choose the most frequent final answer (self-consistency vote)
    counts = Counter(short_answers)
    groups = {s: [] for s in counts}
    for idx, s in enumerate(short_answers):
        groups[s].append(idx)

    mc = counts.most_common()
    if not mc:
        majority_winners, final_answer = [], None
    else:
        top_freq = mc[0][1]
        majority_winners = [s for s, f in mc if f == top_freq]
        final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

In [24]:
results = self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

[Sample 1/5] → '83'
[Sample 2/5] → '22'
[Sample 3/5] → '54'
[Sample 4/5] → '83'
[Sample 5/5] → '83'


In [25]:
print(results["final_answer"])

83


In [26]:
print(results["full_answers"][0])

 To find the value of \( x \), let's solve the equation step by step.

1. **Given Equation:**
   \[
   \text{Half the value of } 3x - 9 \text{ is } x + 37.
   \]
   This can be written as:
   \[
   \frac{1}{2}(3x - 9) = x + 37.
   \]

2. **Multiply Both Sides by 2 to Eliminate the Fraction:**
   \[
   2 \times \frac{1}{2}(3x - 9) = 2 \times (x + 37).
   \]
   Simplifying both sides:
   \[
   3x - 9 = 2x + 74.
   \]

3. **Subtract \( 2x \) from Both Sides to Get the \( x \)-terms on One Side:**
   \[
   3x - 2x - 9 = 74.
   \]
   Simplifying:
   \[
   x - 9 = 74.
   \]

4. **Add 9 to Both Sides to Solve for \( x \):**
   \[
   x - 9 + 9 = 74 + 9.
   \]
   Simplifying:
   \[
   x = 83.
   \]

**Final Answer:**
\[
\boxed{83}
\]


In [27]:
results = self_consistency_vote(
    model,
    tokenizer,
    prompt + "\n\nExplain step by step.",
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

[Sample 1/5] → '83'
[Sample 2/5] → '83'
[Sample 3/5] → '83'
[Sample 4/5] → '83'
[Sample 5/5] → '83'
